# Adaptive Fusion TS2Img — Colab command-only runner

Notebook này chỉ gọi lệnh. Logic nằm trong `src/`; cấu hình nằm trong `config/`. Code được clone về `/content`, còn outputs/cache/checkpoints lưu trên Google Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/adaptive-fusion-ts2img.git"
PROJECT_DIR = "/content/adaptive-fusion-ts2img"
DRIVE_ROOT = "/content/drive/MyDrive/research_ts2img_adaptive_fusion"

In [ ]:
import os, shutil
if not os.path.exists(PROJECT_DIR):
    !git clone {GITHUB_REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

In [ ]:
!pip install -r requirements.txt

## Cập nhật đường dẫn runtime cho Colab

In [ ]:
!python -m src.cli.configure_run \
  --base-config config/default.yaml \
  --out-config config/runtime_colab.yaml \
  --dataset Coffee \
  --image-size 64 \
  --seed 42 \
  --data-root data/UCR \
  --output-root {DRIVE_ROOT}/outputs \
  --cache-root {DRIVE_ROOT}/cache

## Kiểm tra dữ liệu

In [ ]:
!python -m src.cli.check_data --config config/runtime_colab.yaml

## Chạy một thí nghiệm

In [ ]:
!python -m src.train --config config/runtime_colab.yaml

## Chạy theo giai đoạn

Sau khi dữ liệu đã đủ, chạy lần lượt từng stage.

In [ ]:
# Stage 1: smoke test
!python -m src.cli.batch_run --suite config/suites/stage1_smoke_5datasets_3seeds_all_methods.yaml --set paths.output_root={DRIVE_ROOT}/outputs paths.cache_root={DRIVE_ROOT}/cache

In [ ]:
# Gom kết quả và xuất bảng
!python -m src.cli.collect_results --output-root {DRIVE_ROOT}/outputs --out-csv {DRIVE_ROOT}/outputs/summary_all_runs.csv
!python -m src.cli.export_paper_tables --results-csv {DRIVE_ROOT}/outputs/summary_all_runs.csv --out-dir {DRIVE_ROOT}/outputs/paper_tables

In [ ]:
# Kiểm định thống kê sau khi có đủ nhiều dataset/method
!python -m src.cli.rank_results --results-csv {DRIVE_ROOT}/outputs/summary_all_runs.csv --metric test_macro_f1 --out-dir {DRIVE_ROOT}/outputs/paper_tables
!python -m src.cli.statistical_tests --results-csv {DRIVE_ROOT}/outputs/summary_all_runs.csv --metric test_macro_f1 --proposed adaptive_fusion_full --out-dir {DRIVE_ROOT}/outputs/statistical_tests